In [1]:
import numpy as np
import pandas as pd
from scipy.stats import skew
import os
import warnings
warnings.filterwarnings('ignore')

# 1. Load raw data
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

# Drop outlier samples identified in EDA
train = train[~train['Id'].isin([524, 1299])].reset_index(drop=True)

# 2. Extract target and combine features
y_train = np.log1p(train['SalePrice'])
n_train = len(train)
features = pd.concat([train.drop(['Id', 'SalePrice'], axis=1), 
                      test.drop(['Id'], axis=1)], axis=0).reset_index(drop=True)

# 3. Quick missing value imputation (from Day 4)
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']
features[none_cols] = features[none_cols].fillna('None')

mode_cols = ['Electrical', 'KitchenQual', 'Functional', 'SaleType', 'MSZoning', 'Exterior1st', 'Exterior2nd', 'Utilities']
for col in mode_cols:
    features[col] = features[col].fillna(features[col].mode()[0])

features['LotFrontage'] = features.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
features['GarageYrBlt'] = features['GarageYrBlt'].fillna(features['YearBuilt'])

zero_cols = ['GarageCars', 'GarageArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
features[zero_cols] = features[zero_cols].fillna(0)

print(f"Combined features shape after basic cleaning: {features.shape}")

Combined features shape after basic cleaning: (2917, 79)


In [2]:
# --- Ordinal Encoding ---
# Define mapping dictionary
qual_mapping = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
ordinal_cols = ['ExterQual', 'ExterCond', 'KitchenQual', 'HeatingQC', 
                'BsmtQual', 'BsmtCond', 'FireplaceQu', 'GarageQual', 'GarageCond']

# Apply mapping
for col in ordinal_cols:
    features[col] = features[col].map(qual_mapping)

print("Ordinal encoding applied successfully.")

Ordinal encoding applied successfully.


In [3]:
# --- Feature Construction ---

# 1. Total Square Footage (TotalSF)
features['TotalSF'] = features['TotalBsmtSF'] + features['1stFlrSF'] + features['2ndFlrSF']

# 2. Total Bathrooms (TotalBath)
# Full bath counts as 1, half bath counts as 1/2
features['TotalBath'] = features['FullBath'] + features['BsmtFullBath'] + (features['HalfBath'] + features['BsmtHalfBath']) / 2

# 3. Ages (House age and Remodel age at the time of sale)
features['Age'] = features['YrSold'] - features['YearBuilt']
features['RemodAge'] = features['YrSold'] - features['YearRemodAdd']

# 4. Total Porch Area (PorchSF)
features['PorchSF'] = features['OpenPorchSF'] + features['EnclosedPorch'] + features['3SsnPorch'] + features['ScreenPorch']

# 5. Overall Grade (OverallGrade)
features['OverallGrade'] = features['OverallQual'] * features['OverallCond']

# --- Binary Features (0/1) ---
features['HasPool'] = features['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
features['HasGarage'] = features['GarageArea'].apply(lambda x: 1 if x > 0 else 0)
features['HasFireplace'] = features['Fireplaces'].apply(lambda x: 1 if x > 0 else 0)
features['HasBasement'] = features['TotalBsmtSF'].apply(lambda x: 1 if x > 0 else 0)
features['HasSecondFloor'] = features['2ndFlrSF'].apply(lambda x: 1 if x > 0 else 0)
features['HasWoodDeck'] = features['WoodDeckSF'].apply(lambda x: 1 if x > 0 else 0)

print(f"Features shape after construction: {features.shape}")

Features shape after construction: (2917, 91)


In [4]:
# --- Numerical Skewness ---
features['MSSubClass'] = features['MSSubClass'].apply(str)
features['YrSold'] = features['YrSold'].astype(str)
features['MoSold'] = features['MoSold'].astype(str)

numeric_dtypes = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
numeric_features = features.select_dtypes(include=numeric_dtypes).columns

# Calculate skewness
skewed_feats = features[numeric_features].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
skewness = pd.DataFrame({'Skew': skewed_feats})
skewness = skewness[abs(skewness['Skew']) > 0.75]

# Apply log1p
skewed_features = skewness.index
for feat in skewed_features:
    features[feat] = np.log1p(features[feat])

# --- One-Hot Encoding ---
final_features = pd.get_dummies(features).reset_index(drop=True)

# --- Re-split and Save ---
X_train_final = final_features.iloc[:n_train].copy()
X_test_final = final_features.iloc[n_train:].copy()
y_train_df = pd.DataFrame({'SalePrice_log': y_train})

os.makedirs('../data/processed', exist_ok=True)
X_train_final.to_csv('../data/processed/X_train_engineered.csv', index=False)
X_test_final.to_csv('../data/processed/X_test_engineered.csv', index=False)
y_train_df.to_csv('../data/processed/y_train.csv', index=False)

print(f"Final Train Shape: {X_train_final.shape}")
print(f"Final Test Shape: {X_test_final.shape}")
print("All engineered data saved successfully!")

Final Train Shape: (1458, 306)
Final Test Shape: (1459, 306)
All engineered data saved successfully!
